# task_09 Solution

In [1]:

from pathlib import Path
import pandas as pd

base = Path("../../tasks/task_09/data/")


In [2]:
campaigns = pd.read_csv(base / "campaigns.csv")
ad_spend = pd.read_csv(base / "ad_spend.csv")
sessions = pd.read_csv(base / "sessions.csv")
orders = pd.read_csv(base / "orders.csv")

Q1: For each user who placed at least one order, compute their revenue per session. Using first-touch attribution, which channel has the highest average per-user revenue per session? Break ties alphabetically.

In [3]:
sessions["session_date"] = pd.to_datetime(sessions["session_date"])

# Per-user total sessions
user_sessions = sessions.groupby("user_id").size().reset_index(name="n_sessions")

# First-touch channel per user
user_first = sessions.sort_values("session_date").groupby("user_id").first()[["campaign_id"]].reset_index()
user_first = user_first.merge(campaigns[["campaign_id", "channel"]], on="campaign_id").rename(columns={"channel": "first_channel"})

# Per-user total order revenue
order_sess = orders.merge(sessions[["session_id", "user_id"]], on="session_id")
user_revenue = order_sess.groupby("user_id")["order_value"].sum().reset_index()

# Merge: only users who ordered
converting = user_revenue.merge(user_sessions, on="user_id").merge(user_first[["user_id", "first_channel"]], on="user_id")
converting["rev_per_session"] = converting["order_value"] / converting["n_sessions"]

# Average per-user session revenue by first-touch channel
q1 = converting.groupby("first_channel")["rev_per_session"].mean().idxmax()
print(q1)

email


Q2: For each converting user, attribute total order revenue to first-touch channel. Which first-touch channel has the highest total attributed revenue?

In [4]:
# Reuse converting df from Q1 (has user_id, order_value, first_channel)
ft_revenue = converting.groupby("first_channel")["order_value"].sum()
q2 = ft_revenue.idxmax()
print(q2)

display


Q4: How many users had sessions from multiple channels before their first purchase?

In [5]:
sessions_with_channel = sessions.merge(campaigns[["campaign_id", "channel"]], on="campaign_id")
purchase_sessions = sessions.merge(orders[["session_id"]], on="session_id")
first_purchase_date = purchase_sessions.groupby("user_id")["session_date"].min()
users_multi_channel = 0
for user_id, first_date in first_purchase_date.items():
    channels_before = sessions_with_channel[(sessions_with_channel["user_id"] == user_id) & (sessions_with_channel["session_date"] < first_date)]["channel"].nunique()
    if channels_before >= 2:
        users_multi_channel += 1
q4 = str(users_multi_channel)
print(q4)


10
